# 01 — Leaf-wise growth: grow the best leaf first

*Chapter 10 · LightGBM · Notebook 1 of 5*

A decision tree grows by splitting leaves. The question this notebook answers is the one LightGBM
answers differently from the trees you have built so far: **which leaf do you split next?** Chapter 04's
trees and Chapter 09's XGBoost (by default) grow **level by level** — split every leaf at the current
depth before going deeper. LightGBM grows **leaf-wise**, also called **best-first**: it splits the single
leaf whose best split reduces the loss most, wherever that leaf sits in the tree.

You have already met this idea by name, twice — Chapter 08 NB 5 noted that `HistGradientBoosting*` grows
**leaf-wise** (`max_leaf_nodes`), and Chapter 09 NB 4 named XGBoost's `grow_policy='lossguide'`. Here you
**build** it: a small frontier of candidate leaves, expanded greediest-first, by hand — then match it,
exactly, to LightGBM's first tree.

**Prerequisites.** Chapter 04 (a tree splits to reduce impurity); Chapter 09 NB 2 (the structure-score
gain `½G²/(H+λ)`); Chapters 08–09 (the per-row gradient `g = F − y`).

**What you'll be able to do.**
- State the difference between level-wise and leaf-wise growth as a one-line choice of which leaf to pop.
- Build leaf-wise growth by hand and watch it reach a lower training loss for the same number of leaves.
- See why leaf-wise trees grow lopsided — the seed of the overfitting Notebook 2 will tame.
- Match a by-hand leaf-wise tree to LightGBM's single tree, exactly.

## Level-wise vs leaf-wise — the question

Both strategies use the **same** candidate splits (every feature, every threshold) and the **same** gain
to score them. They differ only in **order**:

- **Level-wise (depthwise)** — split every leaf at the current depth, then move one level deeper. The
  tree stays balanced. This is Chapter 04's growth and XGBoost's default.
- **Leaf-wise (best-first)** — keep a frontier of all current leaves; repeatedly split the **one** whose
  best split has the largest gain, wherever it is. The tree can grow deep down one branch and stay shallow
  on another.

That is the whole idea. The consequences — lower training loss per leaf, but lopsided, overfit-prone
trees — fall out of it, and we watch them happen.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from lightgbm import LGBMRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0
rng = np.random.default_rng(SEED)

# A toy with a deliberate ASYMMETRY: where x0 > 0.5 the target is a 5-step staircase
# in x1; where x0 <= 0.5 it is flat. So one branch deserves many splits and the other
# deserves none -- the perfect place to watch leaf-wise spend its budget where the gain is.
n = 240
x0 = rng.uniform(0, 1, n)
x1 = rng.uniform(0, 1, n)
y = np.where(x0 > 0.5, np.floor(x1 * 5), 0.0) + rng.normal(0, 0.05, n)
X = np.column_stack([x0, x1])

F0 = float(y.mean())     # the constant first prediction (ch 08)
g = F0 - y               # gradient of 1/2 (y - F)^2 at F0 (ch 08/09 convention)
h = np.ones(n)           # squared-error curvature
LAMBDA = 0.0             # no L2 here (ch 09 NB 2's lambda, turned off)
print(f"n = {n}   F0 = mean(y) = {F0:.4f}")
print(f"root SSE = {((y - y.mean()) ** 2).sum():.2f}")


## The gain we'll use (reused from Chapter 09, not rebuilt)

To score a split we reuse Chapter 09 NB 2's **structure score**. A node holding rows with gradient sum
`G = Σg` and Hessian sum `H = Σh` has score `−½ G²/(H + λ)`; splitting it into a left and right child
**reduces** the objective by

`gain = score(parent) − score(left) − score(right)`.

For this first tree the gradients are `g = F₀ − y` and the curvatures `h = 1` (squared error), and we set
`λ = 0`. With those, maximising the gain is exactly minimising the training **SSE** — the same criterion
Chapter 04 used. (And each leaf's value is its **mean `y`** — equivalently `F₀ + (−G/H)`, since
Chapter 09's leaf weight `−G/H` equals `mean(y_leaf) − F₀` added back onto `F₀`.) Nothing new here: the
new idea is the *order* in which we apply it.

In [ ]:
def node_score(idx):
    """Structure score of a node: -1/2 G^2/(H+lambda) (ch 09 NB 2), lambda = LAMBDA = 0."""
    G, H = g[idx].sum(), h[idx].sum()
    return -0.5 * G * G / (H + LAMBDA)


def best_split(idx):
    """Best (feature, threshold, gain, left, right) for a node; gain = the score drop."""
    best = (None, None, -np.inf, None, None)
    parent = node_score(idx)
    for f in range(X.shape[1]):
        vals = np.unique(X[idx, f])
        for t in (vals[:-1] + vals[1:]) / 2:
            left = idx[X[idx, f] < t]
            right = idx[X[idx, f] >= t]
            if len(left) and len(right):
                gain = parent - (node_score(left) + node_score(right))
                if gain > best[2]:
                    best = (f, float(t), gain, left, right)
    return best


feat, thr, gain, _, _ = best_split(np.arange(n))
print(f"root's best split: x{feat} < {thr:.3f}   gain = {gain:.2f}")


## The two strategies, as code

The only difference is **which leaf you pop** from the frontier of current leaves; both then push the two
children to the back:

- **Level-wise** pops the *oldest* leaf (a FIFO queue) — so it fills the tree level by level,
  breadth-first.
- **Leaf-wise** pops the leaf with the *largest gain* (a best-first frontier) — so it spends each split
  where it helps most.

Both stop once there are `num_leaves` leaves; one line of code — which leaf to pop — separates them.
(Budgeting by leaf count, level-wise may stop part-way through a level — that is expected.)

In [ ]:
def sse(leaves):
    """Total training SSE for a list of leaf index-arrays (leaf value = mean y)."""
    return float(sum(((y[idx] - y[idx].mean()) ** 2).sum() for idx in leaves))


def grow(order, budget):
    """Grow to `budget` leaves. The ONLY difference is which leaf we pop:
    'leaf' = the largest-gain leaf (best-first); 'level' = the oldest leaf (FIFO)."""
    frontier = [np.arange(n)]   # leaves that may still split, in creation order
    done = []                   # leaves that cannot be split further
    while len(frontier) + len(done) < budget and frontier:
        splits = [best_split(idx) for idx in frontier]
        if order == "leaf":
            k = max(range(len(frontier)), key=lambda j: splits[j][2])   # the largest-gain leaf
        else:
            k = 0                                                       # the oldest leaf (FIFO)
        idx, bs = frontier.pop(k), splits[k]
        if bs[0] is None:                  # no valid split here: retire this leaf
            done.append(idx)
            continue
        frontier += [bs[3], bs[4]]         # left, right -> the back of the frontier
    return frontier + done


budgets = [2, 3, 4, 5, 6, 8]
sse_leaf = [sse(grow("leaf", b)) for b in budgets]
sse_level = [sse(grow("level", b)) for b in budgets]
for b, sl, sv in zip(budgets, sse_leaf, sse_level, strict=True):
    print(f"  {b} leaves:  leaf-wise SSE {sl:8.2f}   level-wise SSE {sv:8.2f}")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(budgets, sse_leaf, "o-", color=COLORS["model"], linewidth=2.2,
        label="leaf-wise (best-first)")
ax.plot(budgets, sse_level, "o-", color=COLORS["muted"], linewidth=2.2,
        label="level-wise (depthwise)")
ax.set_xlabel("number of leaves")
ax.set_ylabel("training SSE")
ax.set_title("same leaf budget, different growth order")
ax.legend()
plt.show()


**Read the figure.** For the same number of leaves, leaf-wise reaches a far lower **training** loss
— at six leaves, SSE ≈ 0.6 versus ≈ 65. Watch *when* each curve drops: level-wise grows breadth-first, so
its first extra split lands on the flat `x0 ≤ 0.5` branch (no gain — the loss sits near 268 until, a split
later, it reaches the structured `x0 > 0.5` branch and falls to ≈ 65). Leaf-wise goes straight for the
high-gain splits in `x0 > 0.5`. (Whether a lower *training* loss helps on held-out data is the question of
**Notebook 2** — leaf-wise's efficiency is also its overfitting risk.)

In [ ]:
LEAF_COLORS = [COLORS["class_a"], COLORS["class_b"], COLORS["class_c"],
               COLORS["class_d"], COLORS["class_e"], COLORS["muted"]]
budget = 6
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, order, title in [(axes[0], "leaf", "leaf-wise"), (axes[1], "level", "level-wise")]:
    leaves = grow(order, budget)
    for k, idx in enumerate(leaves):
        ax.scatter(X[idx, 0], X[idx, 1], s=28, color=LEAF_COLORS[k % len(LEAF_COLORS)],
                   edgecolor=COLORS["text"], linewidth=0.3)
    ax.axvline(0.5, color=COLORS["text"], linestyle=":", linewidth=1)
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")
    ax.set_title(f"{title} ({budget} leaves)")
    print(f"{title:11s} leaf sizes: {sorted(len(idx) for idx in leaves)}")
fig.tight_layout()
plt.show()


**Read the figure.** Each colour is one leaf. Leaf-wise (left) left the flat `x0 ≤ 0.5` region as
**one big leaf** (about 109 points) and carved the structured `x0 > 0.5` region into fine horizontal
bands — exactly the staircase in `x1`. Level-wise (right) split both regions, spending leaves on the flat
side where they buy nothing. "Best-first" follows the **gain**; "level-wise" follows **position**, and on
asymmetric data that difference is large.

## Matching LightGBM

LightGBM grows leaf-wise. To check our by-hand tree against it, we turn **off** every extra so only the
bare mechanism remains: no L2 penalty (`reg_lambda=0`), no minimum split gain (`min_split_gain=0`), no
leaf-size floor (`min_child_samples=1`), and lossless binning (`max_bin` huge, `min_data_in_bin=1`, so
LightGBM considers every midpoint we do). One tree, `learning_rate=1`. With those, its single tree should
be **exactly** our by-hand leaf-wise tree.

(We leave LightGBM's info log visible — it prints a short banner when it fits; that is the engine telling
us what it did.)

In [ ]:
lgbm = LGBMRegressor(
    n_estimators=1, num_leaves=6, learning_rate=1.0, min_child_samples=1,
    min_split_gain=0.0, reg_lambda=0.0, reg_alpha=0.0, min_child_weight=0.0,
    max_bin=100000, min_data_in_bin=1, random_state=SEED,
)
lgbm.fit(X, y)

# by-hand leaf-wise predictions: each point gets its leaf's mean y
leaves_byhand = grow("leaf", 6)
pred_byhand = np.empty(n)
for idx in leaves_byhand:
    pred_byhand[idx] = y[idx].mean()
pred_lgbm = lgbm.predict(X)

print(f"max |pred_byhand - pred_lgbm| = {np.max(np.abs(pred_byhand - pred_lgbm)):.6f}")
print(f"by-hand train SSE {((y - pred_byhand) ** 2).sum():.4f}   "
      f"LightGBM {((y - pred_lgbm) ** 2).sum():.4f}")

fig, ax = plt.subplots(figsize=(5.6, 5.6))
lo, hi = y.min() - 0.2, y.max() + 0.2
ax.plot([lo, hi], [lo, hi], color=COLORS["muted"], linestyle="--", linewidth=1, label="y = x")
ax.scatter(pred_byhand, pred_lgbm, s=40, color=COLORS["model"], edgecolor=COLORS["text"],
           linewidth=0.3, zorder=3)
ax.set_xlabel("by-hand leaf-wise prediction")
ax.set_ylabel("LightGBM prediction")
ax.set_title("by-hand vs LightGBM — identical")
ax.legend()
plt.show()


**Read the figure.** Every point sits on the diagonal: the by-hand leaf-wise tree and LightGBM's
single tree make the **identical** prediction for every row (`max|Δ| = 0`), with the same training SSE.
LightGBM's "best-first" growth *is* the frontier you built — it expands the largest-gain leaf, one at a
time. (The thresholds LightGBM stores differ by a hair from our midpoints — its bin-boundary convention —
but the partition, and therefore every prediction, is the same.)

## Where this sits — and it is not LightGBM-only

Leaf-wise reaches a given training loss with **fewer leaves** than level-wise: it is efficient. But it
grows **deep and lopsided**, pouring splits down whichever branch keeps paying — and an unconstrained
greedy fit is exactly how a model memorizes its training data. That overfitting risk is real, and it is
what **Notebook 2** tames, with the dial LightGBM gives you for it: `num_leaves`.

And leaf-wise is not a LightGBM peculiarity — it is the **modern default**. Scikit-learn's
`HistGradientBoosting*` grows leaf-wise (`max_leaf_nodes=31`, Chapter 08 NB 5) and XGBoost offers it as
`grow_policy='lossguide'` (Chapter 09 NB 4). LightGBM made it the default and built the rest of the
library around it.

## Your turn

1. **(easy)** Change the leaf budget in Figure 1 (say to 10 or 16) and read the leaf-wise vs level-wise
   SSE gap. Does level-wise ever catch up, and at what cost in leaves?
2. **(core)** Write, in one line, the difference between the two strategies (which leaf you pop). Confirm
   from the code that at **two** leaves both build the same tree — why must the first split always agree?
3. **(reach)** Construct a toy where leaf-wise and level-wise build the **same** tree. When does the
   growth order stop mattering? (Hint: think about a target whose high-gain splits happen to lie on
   complete levels — for example, perfectly symmetric structure in both branches.)

## What you built

You built **leaf-wise (best-first) growth** by hand: a frontier of candidate leaves, expanded
largest-gain-first, using Chapter 09's structure-score gain. You saw it reach a lower **training** loss
than level-wise for the same number of leaves — by spending splits where the gain is, which makes the tree
**lopsided** — and you matched it to LightGBM's single tree **exactly**.

**Vocabulary.** leaf-wise / best-first · level-wise / depthwise · the leaf frontier · `num_leaves` (the
leaf budget — the subject of Notebook 2).

Next: **`num_leaves`, the central dial** — how leaf-wise capacity is set (not by `max_depth`), and the
overfitting it causes when you turn it up.

## References

- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* Advances in Neural Information Processing Systems 30 (NeurIPS 2017). (Leaf-wise growth.)
- Shi, H. (2007). *Best-First Decision Tree Learning.* MSc thesis, University of Waikato. (Best-first
  tree induction.)
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (The structure-score gain reused
  here — Chapter 09 NB 2.)

*Previous: Chapter 09 — XGBoost.*  ·  *Next: 02 — num_leaves, the central dial.*